In [1]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from nasch import TrafficCA
from nasch_intersection import IntersectionSystem

plt.rcParams.update({
    "figure.figsize":  (8, 5),
    "font.size":       12,
    "axes.grid":       True,
    "grid.alpha":      0.3,
    "axes.labelsize":  12,
    "axes.titlesize":  13,
    "legend.fontsize": 11,
    "savefig.dpi":     150,
    "savefig.bbox":    "tight",
})

DATA_DIR    = "../data"
FIGURES_DIR = "../figures"
os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)


def save_results(path, **arrays):
    np.savez(path, **arrays)


def load_or_compute(path, compute_fn, force=False):
    """Load `path` if it exists; else compute, save, and return."""
    if force or not os.path.exists(path):
        result = compute_fn()
        np.savez(path, **result)
        return result
    with np.load(path) as f:
        return {k: f[k] for k in f.files}


Failed to read module file 'c:\Users\Bakri\AppData\Local\Programs\Python\Python311\Lib\shlex.py' for module 'shlex': UnicodeDecodeError
Traceback (most recent call last):
  File "C:\Users\Bakri\AppData\Roaming\Python\Python311\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Bakri\AppData\Roaming\Python\Python311\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Bakri\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._bootstra

# Two-road intersection — symmetric-demand validation

This notebook is the baseline for **Option B**: a two-road crossing
with one traffic light per road, mutually exclusive. Option A (the
lights sweep in notebook 02) and Option B are being developed in
parallel; we will pick one to carry forward later.

The intersection model (`IntersectionSystem`) has two
open-boundary single-lane NaSch roads crossing at cell `L // 2`.
When road 1 is green, road 2 is red, and vice versa. The green-time
split is controlled by `eta = T_g1 / T`, with cycle `T` fixed.

## Validation logic

Under **symmetric demand** `p_in_1 = p_in_2`, total throughput vs
green-split `eta` must be a symmetric curve peaked at `eta = 0.5`.
- `J1(eta)` should rise with `eta` (more green time for road 1).
- `J2(eta)` should fall with `eta`, as a mirror of `J1`.
- `J_total(eta) = J1 + J2` should be symmetric about `eta = 0.5` and
  maximised there.

If `J_total(eta)` is not symmetric about `0.5`, the module is wrong —
there is a bias in how the two roads see the crossing, and we stop
and fix it before running any asymmetric experiments.


In [2]:
# Symmetric sweep: p_in_1 = p_in_2 = 0.3, vary eta.
SYM_L         = 500
SYM_V_MAX     = 5
SYM_P_RAND    = 0.3
SYM_P_IN      = 0.3
SYM_T         = 80
SYM_ETAS      = np.round(np.arange(0.1, 0.91, 0.1), 2)   # 9 values
SYM_N_SEEDS   = 5
SYM_SEEDS     = np.arange(SYM_N_SEEDS)
SYM_T_WARMUP  = 5000
SYM_T_MEASURE = 10000


def compute_sym():
    n_etas = len(SYM_ETAS)
    J1 = np.zeros((SYM_N_SEEDS, n_etas))
    J2 = np.zeros((SYM_N_SEEDS, n_etas))
    Jt = np.zeros((SYM_N_SEEDS, n_etas))

    for si, s in enumerate(tqdm(SYM_SEEDS, desc="symmetric sweep")):
        for ei, eta in enumerate(SYM_ETAS):
            np.random.seed(int(s) * 10_000 + ei)
            sim = IntersectionSystem(
                L=SYM_L, v_max=SYM_V_MAX, p_rand=SYM_P_RAND,
                p_in_1=SYM_P_IN, p_in_2=SYM_P_IN,
                T=SYM_T, eta=float(eta),
            )
            sim.warmup(SYM_T_WARMUP)
            j1, j2, jt = sim.run(SYM_T_MEASURE)
            J1[si, ei] = j1
            J2[si, ei] = j2
            Jt[si, ei] = jt

    return dict(
        etas      = SYM_ETAS,
        J1_mean   = J1.mean(axis=0), J1_std = J1.std(axis=0),
        J2_mean   = J2.mean(axis=0), J2_std = J2.std(axis=0),
        Jt_mean   = Jt.mean(axis=0), Jt_std = Jt.std(axis=0),
        L         = SYM_L,
        v_max     = SYM_V_MAX,
        p_rand    = SYM_P_RAND,
        p_in      = SYM_P_IN,
        T         = SYM_T,
        seeds     = SYM_SEEDS,
        T_warmup  = SYM_T_WARMUP,
        T_measure = SYM_T_MEASURE,
    )


sym = load_or_compute(f"{DATA_DIR}/intersection_symmetric.npz", compute_sym)
print("symmetric arrays:", sorted(sym.keys()))


symmetric sweep:   0%|          | 0/5 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots()

etas = sym["etas"]
for key, colour, label in [
    ("J1", "tab:blue",    "J1 (road 1)"),
    ("J2", "tab:orange",  "J2 (road 2)"),
    ("Jt", "tab:green",   "J_total"),
]:
    m = sym[f"{key}_mean"]
    s = sym[f"{key}_std"]
    ax.plot(etas, m, color=colour, linewidth=1.8, label=label)
    ax.fill_between(etas, m - s, m + s, color=colour, alpha=0.2)

ax.axvline(0.5, color="k", linestyle=":", alpha=0.4)
ax.set_xlabel(r"Green-time split $\eta = T_{g1} / T$")
ax.set_ylabel("Throughput J (cars/timestep)")
ax.set_title(f"Symmetric demand (p_in_1 = p_in_2 = {sym['p_in'].item():.2f})")
ax.legend(loc="best")

fig.savefig(f"{FIGURES_DIR}/03_symmetric.png")
plt.show()

# Programmatic symmetry check: J_total should be invariant under eta -> 1 - eta.
Jt_mean = sym["Jt_mean"]
asym    = np.abs(Jt_mean - Jt_mean[::-1]).max()
tol     = 3 * sym["Jt_std"].max()
print(f"Symmetry check: max |J_total(eta) - J_total(1-eta)| = {asym:.4f}")
print(f"                tolerance (3 x max seed-std)        = {tol:.4f}")
if asym > tol:
    print("WARNING: J_total is NOT symmetric about eta=0.5. "
          "The crossing-cell obstacle logic is biased — stop and fix.")
else:
    print("OK: J_total is symmetric about eta=0.5 within seed noise.")


In [ ]:
# Sanity check: at eta = 0.5, each road is green half the time, so its
# throughput should be about half the baseline single-road throughput
# at the same p_in. A large discrepancy means the crossing-cell
# obstacle logic is wrong (e.g. stale timing, wrong cell, off-by-one).

BASE_L, BASE_V_MAX, BASE_P_RAND, BASE_P_IN = 500, 5, 0.3, 0.3
BASE_T_WARMUP, BASE_T_MEASURE              = 5000, 10000
BASE_SEEDS                                 = np.arange(3)

baseline = np.zeros(len(BASE_SEEDS))
for si, s in enumerate(BASE_SEEDS):
    np.random.seed(int(s))
    ca = TrafficCA(L=BASE_L, v_max=BASE_V_MAX,
                   p_rand=BASE_P_RAND, p_in=BASE_P_IN)
    ca.warmup(BASE_T_WARMUP)
    baseline[si] = ca.run(BASE_T_MEASURE)
single_road = baseline.mean()
predicted   = 0.5 * single_road

# Per-road throughput at eta = 0.5 from the sweep above.
idx_half   = int(np.argmin(np.abs(sym["etas"] - 0.5)))
per_road_1 = sym["J1_mean"][idx_half]
per_road_2 = sym["J2_mean"][idx_half]
ratio_1    = per_road_1 / single_road
ratio_2    = per_road_2 / single_road

print(f"Single-road baseline (p_in={BASE_P_IN}): {single_road:.4f} cars/step")
print(f"Predicted per-road at eta=0.5:         {predicted:.4f} "
      f"(= 0.5 x baseline)")
print(f"Observed J1 at eta=0.5:                {per_road_1:.4f} "
      f"(ratio to baseline: {ratio_1:.3f})")
print(f"Observed J2 at eta=0.5:                {per_road_2:.4f} "
      f"(ratio to baseline: {ratio_2:.3f})")

if not (0.4 <= ratio_1 <= 0.6) or not (0.4 <= ratio_2 <= 0.6):
    print("WARNING: per-road/baseline ratio outside [0.4, 0.6] — investigate.")
else:
    print("OK: both per-road ratios are in [0.4, 0.6].")


In [ ]:
# Asymmetric demand sweep: fix eta = 0.5, vary (p_in_1, p_in_2).
# First pass uses half-size warmup/measurement so the heatmap renders
# same-day; rerun with force=True after T_WARMUP/T_MEASURE are raised
# to full spec (5000 / 10000) for the submission figure.
ASYM_L         = 500
ASYM_V_MAX     = 5
ASYM_P_RAND    = 0.3
ASYM_T         = 80
ASYM_ETA       = 0.5
ASYM_P_INS     = np.round(np.arange(0.1, 0.91, 0.1), 2)   # 9 values
ASYM_N_SEEDS   = 5
ASYM_SEEDS     = np.arange(ASYM_N_SEEDS)
ASYM_T_WARMUP  = 2000    # full spec: 5000
ASYM_T_MEASURE = 4000    # full spec: 10000


def compute_asym():
    n = len(ASYM_P_INS)
    J1 = np.zeros((ASYM_N_SEEDS, n, n))
    J2 = np.zeros((ASYM_N_SEEDS, n, n))
    Jt = np.zeros((ASYM_N_SEEDS, n, n))

    total = ASYM_N_SEEDS * n * n
    pbar  = tqdm(total=total, desc="asym (p1, p2, seed)")
    for si, s in enumerate(ASYM_SEEDS):
        for i, p1 in enumerate(ASYM_P_INS):
            for j, p2 in enumerate(ASYM_P_INS):
                np.random.seed(int(s) * 10_000 + i * len(ASYM_P_INS) + j)
                sim = IntersectionSystem(
                    L=ASYM_L, v_max=ASYM_V_MAX, p_rand=ASYM_P_RAND,
                    p_in_1=float(p1), p_in_2=float(p2),
                    T=ASYM_T, eta=ASYM_ETA,
                )
                sim.warmup(ASYM_T_WARMUP)
                j1, j2, jt = sim.run(ASYM_T_MEASURE)
                J1[si, i, j] = j1
                J2[si, i, j] = j2
                Jt[si, i, j] = jt
                pbar.update(1)
    pbar.close()

    return dict(
        p_ins     = ASYM_P_INS,
        J1_mean   = J1.mean(axis=0), J1_std = J1.std(axis=0),
        J2_mean   = J2.mean(axis=0), J2_std = J2.std(axis=0),
        Jt_mean   = Jt.mean(axis=0), Jt_std = Jt.std(axis=0),
        L         = ASYM_L,
        v_max     = ASYM_V_MAX,
        p_rand    = ASYM_P_RAND,
        T         = ASYM_T,
        eta       = ASYM_ETA,
        seeds     = ASYM_SEEDS,
        T_warmup  = ASYM_T_WARMUP,
        T_measure = ASYM_T_MEASURE,
    )


asym = load_or_compute(f"{DATA_DIR}/intersection_asym_demand.npz", compute_asym)
print("asymmetric arrays:", sorted(asym.keys()))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

p_ins = asym["p_ins"]
Jt    = asym["Jt_mean"]

im = ax.imshow(
    Jt,
    origin="lower",
    extent=[p_ins[0], p_ins[-1], p_ins[0], p_ins[-1]],
    aspect="auto",
    cmap="viridis",
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("J_total (cars/timestep)")

# Overlay contour lines at 0.1, 0.2, ... for readability.
P1, P2 = np.meshgrid(p_ins, p_ins, indexing="ij")
levels = np.arange(0.1, Jt.max() + 0.1, 0.1)
cs = ax.contour(P1, P2, Jt, levels=levels, colors="white",
                linewidths=0.8, alpha=0.7)
ax.clabel(cs, inline=True, fontsize=9, fmt="%.1f")

ax.set_xlabel("p_in_1 (road 1 inflow)")
ax.set_ylabel("p_in_2 (road 2 inflow)")
ax.set_title(f"J_total(p_in_1, p_in_2) at eta = {asym['eta'].item():.2f}")
fig.savefig(f"{FIGURES_DIR}/03_asym_heatmap.png")
plt.show()


## Interpretation — asymmetric-demand heatmap

**Predictions.**

- **Undersaturated regime** (low `p_in_1`, low `p_in_2`): every car
  that arrives makes it through the intersection within its light
  window. Total throughput should track total demand:
  `J_total ≈ p_in_1 + p_in_2`. The bottom-left corner of the heatmap
  should show this diagonal structure.
- **Saturated regime** (high `p_in_1`, high `p_in_2`): at `eta = 0.5`
  each road can only flow half the time, so the intersection caps out
  at roughly the single-road capacity (≈ half the baseline throughput
  per road, summed). The top-right corner of the heatmap should
  plateau.
- **Mixed regime** (one high, one low): total throughput is bounded
  by the sum of (saturated road) plus (undersaturated road's demand).
  Contour lines should bend towards the saturated road's axis as that
  demand rises.

**Observations.**

*Record observations from the rendered heatmap here. Flag any
disagreement with the predictions above — in particular, a top-right
plateau that sits noticeably below `2 × (baseline / 2) = baseline`
would indicate the intersection is leaking capacity (e.g. cars
stopped at the crossing are not clearing during green).*
